In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
DB_PATH = REPO_ROOT / "database" / "CSCAF_public.db"
EXPORT_FILE = REPO_ROOT / "raw_data" / "job_advertisements" / "job_advertisements_full.json"
SOURCE_COLLECTION = "JOBS_EU_2025"
REPORT_FOLDER = REPO_ROOT / "data_load" / "load_reports"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

assert DB_PATH.exists(), "run 01_load_ecsf_to_database.ipynb and 02_load_curricula_to_database.ipynb first"

In [ ]:
from cscaf.loading import AdvertisementLoader
from cscaf.storage import Storage
from cscaf.study import Study

study = Study.from_config()
storage = Storage.of(study, read_only=False)
print("database:", storage.path.relative_to(REPO_ROOT))
print("export  :", EXPORT_FILE.relative_to(REPO_ROOT))

In [ ]:
{"country": storage.count_rows("country"),
 "job_advertisement_snapshot": storage.count_rows("job_advertisement_snapshot"),
 "job_advertisement": storage.count_rows("job_advertisement")}

In [ ]:
report = AdvertisementLoader(study, storage).load(EXPORT_FILE, REPORT_FOLDER, SOURCE_COLLECTION)

In [ ]:
problems = report.verify(storage)
assert not problems, problems
print("snapshot  :", {k: report.snapshot.get(k) for k in ("id", "n_in_file", "n_not_eu", "n_duplicates_removed", "n_loaded")})
print("left out  :", report.left_out.get("not_eu"), "| EU countries without ads:", report.left_out.get("eu_countries_without_ads"))
print("dates     :", report.date_range)
print("not in English:", len(report.not_english), "lines in", len({p["ad_id"] for p in report.not_english}), "ads")
assert storage.count_rows("job_advertisement") == 5136
report.ads_per_country